# Notebook 01 — EDA Cơ bản
## Thống kê mô tả, kiểm tra chất lượng dữ liệu, phân bố nhãn

**Dataset:** Predict Students' Dropout and Academic Success — UCI ML Repository  
**Mục tiêu:** Hiểu cấu trúc dữ liệu, kiểm tra missing values, phân bố các đặc trưng.


In [ ]:
import sys
sys.path.insert(0, '..')  # Để import từ src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.data.load_data import load_raw_data, basic_info, check_data_quality, FEATURE_COLS, TARGET_COL

DATA_PATH = '../data/raw/dataset.csv'
print('Thư viện đã import thành công ✅')

## 1. Đọc và kiểm tra dataset

In [ ]:
df = load_raw_data(DATA_PATH)
basic_info(df)

In [ ]:
quality = check_data_quality(df)
print('\nMissing values chi tiết:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
print('5 dòng đầu:')
df.head()

In [ ]:
print('Thống kê mô tả:')
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 2. Phân bố nhãn Target

In [ ]:
PALETTE = {'Dropout': '#e74c3c', 'Enrolled': '#f39c12', 'Graduate': '#2ecc71'}
counts = df[TARGET_COL].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
colors = [PALETTE[l] for l in counts.index]
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{cnt}\n({cnt/len(df)*100:.1f}%)', ha='center', fontsize=11)
axes[0].set_title('Số lượng mẫu theo nhãn', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Số sinh viên')
axes[0].set_ylim(0, counts.max() * 1.2)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Tỷ lệ phân bố nhãn', fontsize=13, fontweight='bold')

plt.suptitle('Phân bố nhãn Target', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Phân bố các đặc trưng số chính

In [ ]:
numeric_key_features = [
    'Age at enrollment', 'Admission grade',
    'Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)',
    'Curricular units 1st sem (approved)', 'Curricular units 2nd sem (approved)',
    'Previous qualification (grade)', 'Unemployment rate',
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feat in enumerate(numeric_key_features):
    for label, color in PALETTE.items():
        subset = df[df[TARGET_COL] == label][feat]
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(feat, fontsize=10)
    axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Phân phối đặc trưng số theo nhãn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Kiểm tra outliers — Boxplot

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(numeric_key_features):
    sns.boxplot(data=df, x=TARGET_COL, y=feat, palette=PALETTE, ax=axes[i])
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel('')

plt.suptitle('Boxplot đặc trưng theo nhãn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Tổng kết EDA cơ bản

In [ ]:
print('=== TÓM TẮT EDA CƠ BẢN ===')
print(f'Tổng số mẫu: {len(df)}')
print(f'Số đặc trưng: {len(FEATURE_COLS)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('Phân bố nhãn:')
for label, cnt in df[TARGET_COL].value_counts().items():
    print(f'  {label:12}: {cnt:5d} ({cnt/len(df)*100:.1f}%)')
print()
print('Nhận xét:')
print('  ✅ Không có missing values — dataset sạch')
print('  ⚠️  Class imbalance: Enrolled chỉ ~18% → cần SMOTE')
print('  📊 Dropout có điểm học kỳ thấp hơn rõ rệt so với Graduate')
print('  📊 Sinh viên Dropout có xu hướng nợ học phí cao hơn')